<a href="https://colab.research.google.com/github/isxd0r4/analiseRotatividadeDeFuncionarios/blob/main/employeeAttritionPerformance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl -L -o /content/ibm-hr-analytics-employee-attrition-performance.zip\
        https://www.kaggle.com/api/v1/datasets/download/uniabhi/ibm-hr-analytics-employee-attrition-performance

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 51314  100 51314    0     0  99771      0 --:--:-- --:--:-- --:--:-- 99771


In [ ]:
!unzip -qq /content/ibm-hr-analytics-employee-attrition-performance.zip
!mv /content/WA_Fn-UseC_-HR-Employee-Attrition.csv /content/ibm-hr-analytics-employee-attrition-performance.csv

In [ ]:
import pandas as pd

employee = pd.read_csv("/content/ibm-hr-analytics-employee-attrition-performance.csv")

print(employee.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize = (10, 5))
sns.countplot(data = employee, x = "OverTime", hue = "Attrition", palette = {"Yes" : "green", "No" : "pink"})
plt.title("Relação entre horas extras e evasão de funcionários")
plt.xlabel("Fez horas extras?")
plt.ylabel("Quantidade de funcionários")
plt.show()

print("\n")
plt.figure(figsize = (10, 5))
sns.countplot(data = employee, x = "YearsAtCompany", hue = "Attrition", palette = {"Yes" : "green", "No" : "pink"})
plt.title("Relação entre tempo na empresa e evasão de funcionários")
plt.xlabel("Quanto tempo ficou na empresa?")
plt.ylabel("Quantidade de funcionários")
plt.show()

print("\n")
plt.figure(figsize = (10, 5))
sns.countplot(data = employee, x = "DistanceFromHome", hue = "Attrition", palette = {"Yes" : "green", "No" : "pink"})
plt.title("Relação entre distância de casa e evasão de funcionários")
plt.xlabel("Distância")
plt.ylabel("Quantidade de funcionários")
plt.show()

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

employee = pd.read_csv("/content/ibm-hr-analytics-employee-attrition-performance.csv")

# remoção de colunas irrelevantes para a análise
colunasIrrelevantes = ["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]
employee = employee.drop(columns = colunasIrrelevantes, errors = "ignore") # errors = "ignore" evita erro caso alguma coluna não exista

# convertendo 'yes' e 'no' para binário
employee["Attrition"] = employee["Attrition"].apply(lambda x: 1 if x == "Yes" else 0)

# transformação de texto para número
employee = pd.get_dummies(employee, drop_first = True)

# separação de variáveis
x = employee.drop("Attrition", axis = 1) # dados que usaremos para prever -> axis = 1 atua nas colunas, ou seja, remove a coluna "Attrition"
y = employee["Attrition"] # o que queremos prever, ou seja, a variável alvo

# divisão dos dados para treino e teste
xTrain, xTest, yTrain, yTest = train_test_split(x, y, test_size = 0.3, random_state = 42, stratify = y)

# padronização da escala
scaler = StandardScaler()
xTrainScaled = scaler.fit_transform(xTrain)
xTestScaled = scaler.transform(xTest)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score

# definição das grades de hiperparâmetros
paramGridLogisticRegression = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear'],
    'class_weight': ['balanced'] # usado pois os dados estão desbalanceados
}

paramGridRandomForest = {
    'n_estimators': [100, 200, 500], # número de árvores na floresta
    'max_depth': [5, 10, 25], # profundidade máxima das árvores
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10, 15],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}

# GridSearchCV priorizando a métrica "recall" e utilizando validação cruzada com 8 divisões
gridLogisticRegression = GridSearchCV(LogisticRegression(random_state = 42), paramGridLogisticRegression, cv = 5, scoring = 'recall')
gridRandomForest = GridSearchCV(RandomForestClassifier(random_state = 42), paramGridRandomForest, cv = 5, scoring = 'recall')

# treinamento dos modelos
gridLogisticRegression.fit(xTrainScaled, yTrain)
gridRandomForest.fit(xTrain, yTrain)

# armazena os melhores modelos
melhorLR = gridLogisticRegression.best_estimator_
melhorRF = gridRandomForest.best_estimator_

# validação cruzada nos dados de treino
cvLogisticRegression = cross_val_score(melhorLR, xTrainScaled, yTrain, cv = 5, scoring = "recall")
cvRandomForest = cross_val_score(melhorRF, xTrain, yTrain, cv = 5, scoring = "recall")

# imprimindo os resultados
print(f"recall - regressão logística: {cvLogisticRegression.mean():.2f}")
print(f"recall - random forest: {cvRandomForest.mean():.2f}")

recall - regressão logística: 0.78
recall - random forest: 0.54


In [ ]:
from sklearn.metrics import classification_report

previsaoLR = melhorLR.predict(xTestScaled)
previsaoRF = melhorRF.predict(xTest)

print("performance nos dados de teste - regressão logística:")
print(classification_report(yTest, previsaoLR))
print("performance nos dados de teste - random forest:")
print(classification_report(yTest, previsaoRF))

performance nos dados de teste - regressão logística:
              precision    recall  f1-score   support

           0       0.94      0.73      0.82       370
           1       0.34      0.75      0.47        71

    accuracy                           0.73       441
   macro avg       0.64      0.74      0.64       441
weighted avg       0.84      0.73      0.76       441

performance nos dados de teste - random forest:
              precision    recall  f1-score   support

           0       0.91      0.87      0.89       370
           1       0.45      0.54      0.49        71

    accuracy                           0.82       441
   macro avg       0.68      0.70      0.69       441
weighted avg       0.83      0.82      0.82       441

